# Geometric Brownian Motion and Lognormal Prices

Brownian motion (10/02) can go negative and grows in absolute, not relative, steps. Stock prices do neither: they stay positive and their
moves scale with their level. Geometric Brownian motion (GBM) models the **log** price as a Brownian motion with drift, which makes the price
lognormal. This one change explains several facts that surprise investors: the expected price grows at $\mu$, the typical (median) price only at
$\mu - \sigma^2/2$; a volatile asset can be expected to grow while most of its paths shrink; and simulating it naively introduces errors. This notebook
derives the lognormal distribution of prices and works through these consequences.

## 0. Math link

- [Normal distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/10_normal_distribution.ipynb):
  the log price is normal; the price is its exponential.
- Brownian motion with drift from [10/02](./02_from_random_walk_to_brownian_motion.ipynb).
- Log returns and continuous compounding ([00/01](../00_returns_and_compounding/01_simple_vs_log_returns.ipynb), [00/02](../00_returns_and_compounding/02_continuous_compounding_as_a_limit.ipynb)); the noise in an estimated mean from
  [06/01](../06_estimation_error/01_how_noisy_is_a_mean_return.ipynb); volatility drag and leverage from
  [05/03](../05_risk_measures/03_drawdowns.ipynb).

## 1. Introduction

A pension fund projects CHF 1 million invested in equities over 30 years with an expected return of 7% and volatility of 20%. The expected
final value is CHF 8.2 million. Yet in half of the scenarios the fund ends with less than CHF 4.5 million, and in 71% with less than the
"expected" 8.2 million. Both numbers are correct; they describe a lognormal distribution, whose mean lies far above its median.

After this notebook you can:

- write GBM and its exact solution, and simulate it without discretization error;
- derive the mean, median, mode and quantiles of a lognormal price;
- explain volatility drag: why the median grows at $\mu - \sigma^2/2$ and when most paths lose money;
- compute probabilities of ending above or below a level (the $N(d)$ terms of Black–Scholes);
- estimate $\mu$ and $\sigma$ from data and see which is reliable;
- compare exact simulation with the Euler scheme.

## 2. Theory

### The model

GBM is written as a stochastic differential equation:

$$
dS_t = \mu S_t\,dt + \sigma S_t\,dW_t
$$

(the return over a short interval, $dS/S$, has mean $\mu\,dt$ and variance $\sigma^2\,dt$). Its solution is

$$
S_t = S_0\exp\Big(\big(\mu - \tfrac12\sigma^2\big)t + \sigma W_t\Big)
$$

10/04 derives this with Itô's lemma. A heuristic: the log return over a short step is $\ln(1 + R) \approx R - R^2/2$ with $E[R] = \mu\,dt$ and $E[R^2] \approx \sigma^2dt$, so the
log price drifts at $\nu = \mu - \sigma^2/2$. The log price is a Brownian motion with drift $\nu$ and volatility $\sigma$.

### Lognormal distribution

$\ln S_t \sim N\big(\ln S_0 + \nu t,\ \sigma^2t\big)$, so $S_t$ is lognormal with

$$
\operatorname{median}(S_t) = S_0e^{\nu t}, \qquad E[S_t] = S_0e^{\mu t}, \qquad \operatorname{mode}(S_t) = S_0e^{(\mu - \frac32\sigma^2)t}
$$

$$
\operatorname{Var}(S_t) = S_0^2e^{2\mu t}\big(e^{\sigma^2t} - 1\big)
$$

The mean follows from the normal moment-generating function: $E[e^X] = e^{m + s^2/2}$ for $X \sim N(m, s^2)$. Mode < median < mean: the distribution is skewed to the right, more so
as $\sigma^2t$ grows.

### Probabilities and quantiles

$$
P(S_T > K) = \Phi\left(\frac{\ln(S_0/K) + \nu T}{\sigma\sqrt T}\right), \qquad
q_\alpha(S_T) = S_0\exp\big(\nu T + z_\alpha\sigma\sqrt T\big)
$$

With $\mu$ replaced by the risk-free rate $r$, the first expression is the $N(d_2)$ of Black–Scholes (10/05).

### Volatility drag

The median grows at $\nu = \mu - \sigma^2/2$, the mean at $\mu$. If $\mu < \sigma^2/2$, the median falls, and $P(S_T < S_0) = \Phi(-\nu\sqrt T/\sigma) \to 1$ as $T \to \infty$, although $E[S_T]$ grows: the mean is
carried by a shrinking fraction of very good paths. For a constant leverage $L$ continuously rebalanced (05/03), $\mu_L = r + L(\mu - r)$, $\sigma_L = L\sigma$, and the growth rate
$r + L(\mu - r) - L^2\sigma^2/2$ peaks at $L^* = (\mu - r)/\sigma^2$.

### Simulation

**Exact**: $S_{t+\Delta t} = S_t\exp\big(\nu\Delta t + \sigma\sqrt{\Delta t}\,Z\big)$ with $Z \sim N(0, 1)$, for any step size. **Euler**: $S_{t+\Delta t} = S_t\big(1 + \mu\Delta t + \sigma\sqrt{\Delta t}\,Z\big)$, a first-order
approximation that can produce negative prices and has a distribution error that shrinks only as $\Delta t \to 0$.

### Estimation

From $n$ log returns over $T$ years: $\hat\sigma$ = standard deviation of log returns × $\sqrt{\text{periods per year}}$, precise (06/04); $\hat\nu$ = mean log return per year,
with standard error $\sigma/\sqrt T$ (06/01); $\hat\mu = \hat\nu + \hat\sigma^2/2$.

### Limitations

Constant volatility, no jumps, normal log returns. Real returns have volatility clustering (09/02) and fat tails; GBM is the reference model, not the truth.

## 3. Worked Examples

$S_0 = 100$ (CHF), $\mu = 7\%$, $\sigma = 20\%$, so $\nu = 7\% - 2\% = 5\%$.

### Example 1: ten years

$E[S_{10}] = 100e^{0.7} = 201.4$. Median $100e^{0.5} = 164.9$. Standard deviation of $\ln S_{10}$: $0.2\sqrt{10} = 0.632$.

### Example 2: losing money

$P(S_{10} < 100) = \Phi(-0.5/0.632) = \Phi(-0.791) = 21.4\%$.

### Example 3: a 95% range

$100\exp(0.5 \pm 1.96\cdot 0.632) = 100e^{0.5 \pm 1.240}$: from 47.7 to 570.

### Example 4: volatility drag

With $\sigma = 40\%$ and the same $\mu = 7\%$: $\nu = 7\% - 8\% = -1\%$. The expected price still grows at 7%, but the median falls by 1% a year, and after 30 years
$P(S_{30} < S_0) = \Phi(0.01\sqrt{30}/0.4) = \Phi(0.137) = 55\%$.

### Example 5: the pension projection

$S_0 = 1$ million, 30 years: mean $e^{2.1} = 8.17$ million, median $e^{1.5} = 4.48$ million. The probability of ending below the mean:
$\Phi\big((2.1 - 1.5)/(0.2\sqrt{30})\big) = \Phi(0.548) = 70.8\%$.

## 4. Finance Application

### Where it matters

- **Long-horizon projections.** Pension and wealth-planning tools should report medians and percentiles, not only expected values.
- **Option pricing.** Black–Scholes (10/05) assumes GBM; the probabilities $N(d_1)$, $N(d_2)$ are lognormal probabilities.
- **Leverage and leveraged ETFs.** Daily-rebalanced leveraged products lose value through volatility drag in volatile, trendless markets.
- **Monte Carlo risk.** Simulating prices with the exact lognormal step avoids negative prices and step-size bias.

### The usual misreadings

- "The expected return is 7%, so the typical investor earns 7% a year." The typical (median) investor earns $\mu - \sigma^2/2$ per year, compounded.
- "Expected wealth grows, so the probability of losing money falls over time." Only if $\mu > \sigma^2/2$; otherwise it rises toward 1.
- "Arithmetic average return × years = compound return." The geometric average is lower by about $\sigma^2/2$.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

### Exact simulation and the lognormal moments

In [ ]:
def gbm_paths(S0, mu, sigma, T, n_steps, n_paths, rng_):
    dt = T / n_steps
    incr = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * rng_.standard_normal((n_paths, n_steps))
    return S0 * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(incr, axis=1)], axis=1))

S0, mu, sigma, T = 100.0, 0.07, 0.20, 10.0
nu = mu - 0.5 * sigma**2
P = gbm_paths(S0, mu, sigma, T, 120, 50_000, rng)          # monthly steps, 10 years
ST = P[:, -1]

theory = {"mean": S0 * np.exp(mu * T), "median": S0 * np.exp(nu * T),
          "std": S0 * np.exp(mu * T) * np.sqrt(np.exp(sigma**2 * T) - 1),
          "P(S_T < S_0)": stats.norm.cdf(-nu * T / (sigma * np.sqrt(T))),
          "5% quantile": S0 * np.exp(nu * T + stats.norm.ppf(0.05) * sigma * np.sqrt(T)),
          "95% quantile": S0 * np.exp(nu * T + stats.norm.ppf(0.95) * sigma * np.sqrt(T))}
simulated = {"mean": ST.mean(), "median": np.median(ST), "std": ST.std(), "P(S_T < S_0)": np.mean(ST < S0),
             "5% quantile": np.quantile(ST, 0.05), "95% quantile": np.quantile(ST, 0.95)}
pd.DataFrame({"theory": theory, "simulated": simulated}).round(3)

Simulation and formulas agree: after ten years the mean is CHF 201, the median CHF 165, and the price is below its start in 21% of the paths. The 90%
range runs from CHF 58 to CHF 467, far from symmetric around the mean.

### Volatility drag

In [ ]:
rows = []
for sig in [0.10, 0.20, 0.30, 0.40, 0.60]:
    nu_s = mu - 0.5 * sig**2
    rows.append({"sigma": sig, "median growth nu": nu_s,
                 "P(S < S0) after 1y": stats.norm.cdf(-nu_s / sig),
                 "after 10y": stats.norm.cdf(-nu_s * np.sqrt(10) / sig),
                 "after 30y": stats.norm.cdf(-nu_s * np.sqrt(30) / sig),
                 "E[S_30]/S0": np.exp(mu * 30), "median S_30/S0": np.exp(nu_s * 30)})
pd.DataFrame(rows).set_index("sigma").round(3)

In [ ]:
r_f = 0.01
L = np.linspace(0, 4, 401)
growth = r_f + L * (mu - r_f) - 0.5 * L**2 * sigma**2
L_star = (mu - r_f) / sigma**2
print(f"growth-optimal leverage (mu - r) / sigma^2 = {L_star:.2f}, median growth at L*: {r_f + L_star * (mu - r_f) - 0.5 * L_star**2 * sigma**2:.2%}")
print(f"median growth at L = 1: {r_f + (mu - r_f) - 0.5 * sigma**2:.2%};  at 2 L*: {r_f + 2 * L_star * (mu - r_f) - 2 * L_star**2 * sigma**2:.2%}")

With the expected return fixed at 7%, raising volatility leaves the mean after 30 years unchanged (8.2 times the start) but lowers the median from 7.0 times (10% volatility)
to 0.74 times (40%) and 0.04 times (60%). At 40% volatility the price is more likely to be below its start after 30 years than above it. A constantly levered position
grows fastest at $L^* = 1.5$ (median growth 5.5% against 5.0% unlevered); at twice that leverage the median growth falls to the risk-free 1%.

### Estimating $\mu$ and $\sigma$ from ten years of daily data

In [ ]:
est = []
for _ in range(2_000):
    lr = nu / 252 + sigma / np.sqrt(252) * rng.standard_normal(2_520)          # daily log returns
    s_hat = lr.std(ddof=1) * np.sqrt(252)
    nu_hat = lr.mean() * 252
    est.append((s_hat, nu_hat, nu_hat + 0.5 * s_hat**2))
est = np.array(est)
pd.DataFrame({"true": [sigma, nu, mu], "mean estimate": est.mean(axis=0), "SD of estimate": est.std(axis=0)},
             index=["sigma", "nu (median growth)", "mu"]).round(4)

Ten years of daily data pin $\sigma$ down to ±0.3 points; the median growth rate and $\mu$ have standard errors of 6.2 points (06/01). In GBM the volatility is
the reliable parameter, the drift is not.

### Exact vs Euler simulation

In [ ]:
def euler_paths(S0, mu, sigma, T, n_steps, Z):
    dt = T / n_steps
    S = np.full(Z.shape[0], S0)
    for i in range(n_steps):
        S = S * (1 + mu * dt + sigma * np.sqrt(dt) * Z[:, i])
    return S

def exact_from_Z(S0, mu, sigma, T, n_steps, Z):
    dt = T / n_steps
    return S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * np.sqrt(dt) * Z.sum(axis=1))

rows = []
for sig_e, n_steps in [(0.20, 1), (0.20, 12), (0.20, 252), (0.80, 1), (0.80, 12), (0.80, 252)]:
    Z = rng.standard_normal((20_000, n_steps))
    eu = euler_paths(S0, mu, sig_e, 1.0, n_steps, Z)
    ex = exact_from_Z(S0, mu, sig_e, 1.0, n_steps, Z)
    rows.append({"sigma": sig_e, "steps per year": n_steps, "E[S_1] Euler": eu.mean(), "E[S_1] exact": ex.mean(),
                 "theory": S0 * np.exp(mu), "share negative (Euler)": np.mean(eu < 0),
                 "mean |Euler - exact| (same shocks)": np.mean(np.abs(eu - ex)),
                 "median Euler": np.median(eu), "median theory": S0 * np.exp(mu - 0.5 * sig_e**2)})
pd.DataFrame(rows).set_index(["sigma", "steps per year"]).round(3)

Both schemes get the mean about right, because Euler's one-step return has the correct mean. The distribution is a different matter: with one
step per year and 80% volatility, 9.2% of the Euler prices are negative and the Euler median (107) is far from the lognormal median (78). With 12 or more steps the
medians agree, but individual paths still differ from the exact ones by an average of CHF 11 at 80% volatility with monthly steps.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
t = np.linspace(0, T, P.shape[1])
for i in range(30):
    ax.plot(t, P[i], color="grey", alpha=0.3, linewidth=0.6)
ax.fill_between(t, np.quantile(P, 0.05, axis=0), np.quantile(P, 0.95, axis=0), color="tab:blue", alpha=0.15, label="5–95% of paths")
ax.plot(t, S0 * np.exp(mu * t), color="tab:red", label="mean S0 e^(mu t)")
ax.plot(t, S0 * np.exp(nu * t), color="tab:blue", label="median S0 e^((mu - sigma²/2) t)")
ax.set_ylim(0, 600)
ax.set_title("Geometric Brownian motion: mu = 7%, sigma = 20%, S0 = CHF 100")
ax.set_xlabel("years")
ax.set_ylabel("price (CHF)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.linspace(1, 700, 700)
pdf = stats.lognorm.pdf(x, s=sigma * np.sqrt(T), scale=S0 * np.exp(nu * T))
ax.hist(ST, bins=np.linspace(0, 700, 141), density=True, histtype="step", linewidth=1.2, label="simulated S_10")
ax.plot(x, pdf, color="black", label="lognormal density")
for val, lab, c in [(S0 * np.exp((mu - 1.5 * sigma**2) * T), "mode", "tab:green"), (S0 * np.exp(nu * T), "median", "tab:blue"),
                    (S0 * np.exp(mu * T), "mean", "tab:red")]:
    ax.axvline(val, color=c, linestyle="--", label=f"{lab} {val:.0f}")
ax.set_title("Distribution of the price after 10 years")
ax.set_xlabel("S_10 (CHF)")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
tt = np.linspace(0.01, 50, 500)
for sig, c in [(0.20, "tab:blue"), (0.30, "tab:orange"), (0.40, "tab:red"), (0.60, "tab:purple")]:
    nu_s = mu - 0.5 * sig**2
    axes[0].plot(tt, stats.norm.cdf(-nu_s * np.sqrt(tt) / sig), color=c, label=f"sigma {sig:.0%} (nu = {nu_s:+.1%})")
axes[0].axhline(0.5, color="black", linewidth=0.8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].set_title("P(price below its start), mu = 7%")
axes[0].set_xlabel("years")
axes[0].legend()
axes[1].plot(L, growth, color="tab:blue")
axes[1].axvline(L_star, color="black", linestyle="--", label=f"L* = {L_star:.2f}")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].set_title("Median growth rate of a constantly levered position")
axes[1].set_xlabel("leverage L")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. $S_0 = 50$, $\mu = 6\%$, $\sigma = 15\%$. Compute the mean, median and 90% range of $S_5$.
2. $S_0 = 100$, $\mu = 8\%$, $\sigma = 25\%$. Compute $P(S_1 > 120)$ and $P(S_1 < 80)$.
3. A wealth-planning report states: "Your CHF 1 million portfolio is expected to be worth CHF 8.2 million in 30 years" (7% expected return, 20% volatility). **Interpret** this
   for the client: what is the median outcome, and how likely is the client to end below the stated figure?

### Intermediate

4. Derive $E[S_t] = S_0e^{\mu t}$ from the normal moment-generating function, and $\operatorname{Var}(S_t)$ from $E[S_t^2]$.
5. A buy-and-hold portfolio starts 50/50 in two uncorrelated GBM assets ($\mu = 7\%$, $\sigma = 20\%$ each). Is its value after 20 years lognormal? Compare the skewness of
   its log value with that of a single asset, and compare its median with the median of a continuously rebalanced 50/50 portfolio (a GBM with $\sigma = 20\%/\sqrt2$).

### Advanced

6. For a constantly levered position with $r = 1\%$, $\mu = 7\%$, $\sigma = 20\%$, find the leverage at which the median growth rate becomes negative. Compare with the growth-optimal
   leverage and with the result of 05/03.
7. Strong error of the Euler scheme: for the same Brownian shocks, compute $E|S_T^{\text{Euler}} - S_T^{\text{exact}}|$ for 2, 4, 8, …, 512 steps per year and estimate the convergence order from a log-log
   regression.

## 8. Solutions / Checks

In [ ]:
# 1.
m1, s1, nu1 = 0.06, 0.15, 0.06 - 0.5 * 0.15**2
print(f"1. mean {50 * np.exp(m1 * 5):.2f}, median {50 * np.exp(nu1 * 5):.2f}, 90% range "
      f"{50 * np.exp(nu1 * 5 - 1.645 * s1 * np.sqrt(5)):.2f} to {50 * np.exp(nu1 * 5 + 1.645 * s1 * np.sqrt(5)):.2f}")

# 2.
nu2 = 0.08 - 0.5 * 0.25**2
print(f"2. P(S1 > 120) = {stats.norm.sf((np.log(1.2) - nu2) / 0.25):.4f}, P(S1 < 80) = {stats.norm.cdf((np.log(0.8) - nu2) / 0.25):.4f}")

# 3.
print(f"3. mean {np.exp(0.07 * 30):.2f}m, median {np.exp(0.05 * 30):.2f}m, P(below mean) {stats.norm.cdf((2.1 - 1.5) / (0.2 * np.sqrt(30))):.1%}")

In [ ]:
# 5.
A = gbm_paths(0.5, 0.07, 0.20, 20, 20, 50_000, rng)[:, -1]
B = gbm_paths(0.5, 0.07, 0.20, 20, 20, 50_000, rng)[:, -1]
bh = A + B
single = 2 * A
sig_reb = 0.20 / np.sqrt(2)
print(f"5. skewness of log value: buy-and-hold {stats.skew(np.log(bh)):.3f}, single asset {stats.skew(np.log(single)):.3f}")
print(f"   median: buy-and-hold {np.median(bh):.3f}, rebalanced {np.exp((0.07 - 0.5 * sig_reb**2) * 20):.3f}, single asset {np.median(single):.3f}")

# 6.
a_, b_, c_ = -0.5 * 0.2**2, 0.06, 0.01
L_zero = (-b_ - np.sqrt(b_**2 - 4 * a_ * c_)) / (2 * a_)
print(f"6. median growth negative above L = {L_zero:.2f}; growth-optimal L* = {0.06 / 0.04:.2f}")

# 7.
steps = [2**k for k in range(1, 10)]
errs = []
Zfine = rng.standard_normal((5_000, 512))
for n_s in steps:
    Zc = Zfine.reshape(5_000, n_s, 512 // n_s).sum(axis=2) / np.sqrt(512 // n_s)     # coarser normals from the same path
    errs.append(np.mean(np.abs(euler_paths(S0, mu, sigma, 1.0, n_s, Zc) - exact_from_Z(S0, mu, sigma, 1.0, n_s, Zc))))
slope = np.polyfit(np.log(1 / np.array(steps)), np.log(errs), 1)[0]
print(f"7. strong error from {errs[0]:.3f} (2 steps) to {errs[-1]:.4f} (512 steps); estimated order {slope:.2f}")

**1.** Mean $50e^{0.30} = 67.5$; median $50e^{0.244} = 63.8$; 90% range $50\exp(0.244 \pm 1.645\cdot 0.335)$, from about 36.8 to 110.8.

**2.** $\nu = 8\% - 3.125\% = 4.875\%$. $P(S_1 > 120) = 1 - \Phi((\ln 1.2 - 0.04875)/0.25) = 1 - \Phi(0.534) = 0.297$; $P(S_1 < 80) = \Phi((\ln 0.8 - 0.04875)/0.25) = \Phi(-1.088) = 0.138$.

**3.** CHF 8.2 million is the mean, pulled up by a few very good scenarios. The median outcome is CHF 4.5 million, and the client ends below CHF 8.2 million with probability
about 71%. A projection for a client should show the median and a range (e.g. 10th–90th percentile), and say that the expected value is not what to expect in a typical outcome.

**4.** $\ln S_t \sim N(m, s^2)$ with $m = \ln S_0 + \nu t$, $s^2 = \sigma^2t$; $E[e^X] = e^{m + s^2/2} = S_0e^{\nu t + \sigma^2t/2} = S_0e^{\mu t}$. $E[S_t^2] = E[e^{2X}] = e^{2m + 2s^2} = S_0^2e^{2\mu t + \sigma^2t}$, so $\operatorname{Var} = S_0^2e^{2\mu t}(e^{\sigma^2t} - 1)$.

**5.** The sum of two lognormals is not lognormal: its log value has positive skew (the better asset comes to dominate). Its median lies between the single asset's and that
of the rebalanced portfolio, which, being a GBM with lower volatility, has the highest median growth: diversification with rebalancing reduces volatility drag.

**6.** Median growth $0.01 + 0.06L - 0.02L^2 = 0$ gives $L = 3.16$; the growth-optimal leverage is $L^* = 1.5$, the same Kelly fraction as in 05/03. Beyond $2L^*$ the median growth rate is
below the risk-free rate, and beyond 3.16 it is negative.

**7.** The strong error falls from 1.61 with 2 steps to 0.11 with 512 steps; the log-log slope is 0.49, so the error is proportional to $\sqrt{\Delta t}$. Euler converges slowly path by
path; the exact lognormal step has no discretization error at all.

## 9. Conclusion

- GBM, $dS = \mu S\,dt + \sigma S\,dW$, has the solution $S_t = S_0\exp((\mu - \sigma^2/2)t + \sigma W_t)$: the log price is Brownian motion with drift $\nu = \mu - \sigma^2/2$.
- Prices are lognormal: mean $S_0e^{\mu t}$, median $S_0e^{\nu t}$, mode lower still.
- Volatility drag: the typical path grows at $\nu$; if $\mu < \sigma^2/2$, most paths lose value in the long run while the mean grows.
- $P(S_T > K) = \Phi\big((\ln(S_0/K) + \nu T)/(\sigma\sqrt T)\big)$, the $N(d_2)$ of Black–Scholes when $\mu = r$.
- $\sigma$ is estimated precisely, $\mu$ is not. Simulate with the exact lognormal step, not Euler.

Next: [10/04 Itô's lemma, intuitively](./04_itos_lemma_intuitively.ipynb).